In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType,StructField, StringType, IntegerType

spark = SparkSession.builder.appName("Jupyter").getOrCreate()

conf = spark.sparkContext.getConf()

# Filter and print configurations that start with 'spark.sql'
print("Spark SQL Configurations:")
for key, value in conf.getAll():
    if key.startswith("spark.sql"):
        print(f"{key} = {value}")

print(f"spark.sql.catalog.data.s3.endpoint: {spark.conf.get('spark.sql.catalog.data.s3.endpoint', 'Not Set')}")

for key, value in conf.getAll():
    if key.startswith("spark.hadoop"):
        print(f"{key} = {value}")

spark

Spark SQL Configurations:
spark.sql.catalog.data.s3.endpoint = http://minio-s3:9000
spark.sql.defaultCatalog = data
spark.sql.catalog.kxudata.type = hadoop
spark.sql.catalog.data.warehouse = s3://kxu-iceberg-data
spark.sql.catalog.data.jdbc.password = kxuiceberg
spark.sql.catalog.data.io-impl = org.apache.iceberg.aws.s3.S3FileIO
spark.sql.catalog.data = org.apache.iceberg.spark.SparkCatalog
spark.sql.catalog.data.catalog-impl = org.apache.iceberg.jdbc.JdbcCatalog
spark.sql.catalog.data.jdbc.user = kxuiceberg
spark.sql.catalogImplementation = in-memory
spark.sql.catalog.kxudata.warehouse = /home/iceberg/warehouse
spark.sql.catalog.kxudata = org.apache.iceberg.spark.SparkCatalog
spark.sql.extensions = org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions
spark.sql.warehouse.dir = file:/home/iceberg/notebooks/spark-warehouse
spark.sql.catalog.data.uri = jdbc:postgresql://pg-catalog:5432/kxuiceberg
spark.sql.catalog.data.s3.endpoint: http://minio-s3:9000


25/02/20 20:52:51 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [2]:
data = [("James","","Smith","36636","M",3000),
    ("Michael","Rose","","40288","M",4000),
    ("Robert","","Williams","42114","M",4000),
    ("Maria","Anne","Jones","39192","F",4000),
    ("Jen","Mary","Brown","","F",-1)
  ]

schema = StructType([ \
    StructField("firstname",StringType(),True), \
    StructField("middlename",StringType(),True), \
    StructField("lastname",StringType(),True), \
    StructField("id", StringType(), True), \
    StructField("gender", StringType(), True), \
    StructField("salary", IntegerType(), True) \
  ])

data

df = spark.createDataFrame(data=data, schema=schema)
df.printSchema()


root
 |-- firstname: string (nullable = true)
 |-- middlename: string (nullable = true)
 |-- lastname: string (nullable = true)
 |-- id: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- salary: integer (nullable = true)



In [3]:
df.writeTo("db.test").createOrReplace()

In [4]:
df.writeTo("ketest").createOrReplace()

In [7]:
res = spark.sql("SELECT * FROM db.test")
res.show()

+---------+----------+--------+-----+------+------+
|firstname|middlename|lastname|   id|gender|salary|
+---------+----------+--------+-----+------+------+
|    James|          |   Smith|36636|     M|  3000|
|  Michael|      Rose|        |40288|     M|  4000|
|   Robert|          |Williams|42114|     M|  4000|
|    Maria|      Anne|   Jones|39192|     F|  4000|
|      Jen|      Mary|   Brown|     |     F|    -1|
+---------+----------+--------+-----+------+------+



In [8]:
res = spark.sql("SELECT * FROM db.test")
res.show()

+---------+----------+--------+-----+------+------+
|firstname|middlename|lastname|   id|gender|salary|
+---------+----------+--------+-----+------+------+
|    James|          |   Smith|36636|     M|  3000|
|  Michael|      Rose|        |40288|     M|  4000|
|   Robert|          |Williams|42114|     M|  4000|
|    Maria|      Anne|   Jones|39192|     F|  4000|
|      Jen|      Mary|   Brown|     |     F|    -1|
+---------+----------+--------+-----+------+------+



25/02/20 21:16:53 ERROR StandaloneSchedulerBackend: Application has been killed. Reason: Master removed our application: KILLED
25/02/20 21:16:53 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exiting due to error from cluster scheduler: Master removed our application: KILLED
	at org.apache.spark.errors.SparkCoreErrors$.clusterSchedulerError(SparkCoreErrors.scala:291)
	at org.apache.spark.scheduler.TaskSchedulerImpl.error(TaskSchedulerImpl.scala:981)
	at org.apache.spark.scheduler.cluster.StandaloneSchedulerBackend.dead(StandaloneSchedulerBackend.scala:165)
	at org.apache.spark.deploy.client.StandaloneAppClient$ClientEndpoint.markDead(StandaloneAppClient.scala:263)
	at org.apache.spark.deploy.client.StandaloneAppClient$ClientEndpoint$$anonfun$receive$1.applyOrElse(StandaloneAppClient.scala:170)
	at org.apache.spark.rpc.netty.Inbox.$anonfun$process$1(Inbox.scala:115)
	at org.apache.spark.rpc.netty.Inbox.safelyCall(Inbox.scala:213)
	at org.apache.spark.rpc.netty.Inbox.proce

In [ ]:
spark.stop()